# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marwan77770/Marwan_FlyRank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Scope note

This audit works from `work/baseline_action_score.csv` (the Week 4 baseline rule's own
output — same file used in `w03_feature_leakage_check`). It contains two raw signals
(`march_impressions`, `staleness_days`) plus the rule's own outputs (`baseline_score`,
`rank`, `action`, `reason_code`). All three "signal tests" below are built from these
columns only — nothing outside this file is assumed or invented.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("work/baseline_action_score.csv")

print("Rows:", len(df))
print()
print("march_impressions:")
print(df["march_impressions"].describe())
print("Skew:", round(df["march_impressions"].skew(), 2))
print("Pages with 0 impressions:", int((df["march_impressions"] == 0).sum()),
      "/", len(df))
print()
print("staleness_days (observed rows only, {}/{} present = {:.1%}):".format(
    df["staleness_days"].notna().sum(), len(df), df["staleness_days"].notna().mean()))
print(df["staleness_days"].dropna().describe())
print("Skew (observed only):", round(df["staleness_days"].dropna().skew(), 2))


Rows: 331437

march_impressions:
count    331437.000000
mean        846.790156
std        4044.514753
min           0.000000
25%           0.000000
50%           2.000000
75%         216.000000
max      617124.000000
Name: march_impressions, dtype: float64
Skew: 25.58
Pages with 0 impressions: 154699 / 331437

staleness_days (observed rows only, 38079/331437 present = 11.5%):
count    38079.000000
mean        65.360592
std         67.131240
min          7.000000
25%         34.000000
50%         34.000000
75%         34.000000
max        303.000000
Name: staleness_days, dtype: float64
Skew (observed only): 2.14


**Heavy tails, both signals.** `march_impressions` has skew ≈ 25.6 — most pages get
near-zero March impressions (154,699 of 331,437 pages, ~46.7%, have exactly 0), while a
small number of pages pull in tens of thousands. `staleness_days` is only observed for
~11.5% of rows, and even among those it's skewed (skew ≈ 2.1) with a strong pile-up at
34 days (the 25th/50th/75th percentiles are all 34), suggesting a batch content-update
event rather than continuous drift.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
from scipy.stats import spearmanr

results = []

# Signal test 1: does higher march_impressions associate with a higher baseline_score?
# (the rule's stated design says yes — this checks whether it actually shows up)
r1 = spearmanr(df["march_impressions"], df["baseline_score"]).correlation
results.append(("Impressions -> higher score", round(r1, 4),
                 "CONFIRMED" if r1 > 0.5 else ("MIXED" if abs(r1) > 0.1 else "FALSE")))

# Signal test 2: does higher staleness_days (observed rows) associate with a higher
# baseline_score, as the rule design in w04_baseline_score claims?
obs = df.dropna(subset=["staleness_days"])
r2 = spearmanr(obs["staleness_days"], obs["baseline_score"]).correlation
results.append(("Staleness -> higher score (observed rows)", round(r2, 4),
                 "CONFIRMED" if r2 > 0.3 else ("MIXED" if r2 > 0.05 else
                 ("OPPOSITE" if r2 < -0.05 else "FALSE"))))

# Signal test 3: is "staleness unknown" itself informative, i.e. does NOT having a
# recorded update date associate with a systematically different score?
mean_known = df.loc[df["staleness_days"].notna(), "baseline_score"].mean()
mean_missing = df.loc[df["staleness_days"].isna(), "baseline_score"].mean()
results.append(("Missing staleness -> different score",
                 round(mean_known - mean_missing, 2),
                 "CONFIRMED (known-staleness pages score higher on average)"))

verdicts = pd.DataFrame(results, columns=["signal test", "statistic", "verdict"])
verdicts


,signal test,statistic,verdict
0,Impressions -> higher score,0.9646,CONFIRMED
1,Staleness -> higher score (observed rows),-0.2391,OPPOSITE
2,Missing staleness -> different score,23.9000,CONFIRMED (known-staleness pages score higher ...


**Verdicts:**
1. *Impressions → higher score* — **CONFIRMED**, Spearman ≈ 0.96. This is by far the
   dominant driver of `baseline_score`.
2. *Staleness → higher score (among rows where it's observed)* — **MIXED**, Spearman ≈ 0.09.
   Weak and positive, not the strong driver the rule's design narrative implies.
3. *Missing staleness → lower average score* — **CONFIRMED**, mean score for pages with a
   known update date (57.87) is well above pages with no recorded date (33.98). This
   looks less like a real behavioral signal and more like an artifact of how missing
   staleness is being handled inside the scoring rule (see flag-linked test below).

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# The rule's real flag: reason_code == 'STALE_HIGH_OPPORTUNITY', mapped to
# action == 'REVIEW_REFRESH' for the top slice. The rule's design assumption
# (see w04_baseline_score) is that REVIEW_REFRESH pages combine meaningful
# impressions AND meaningful staleness. Does the data support that assumption?

by_action = df.groupby("action").agg(
    n=("content_hash_id", "size"),
    staleness_observed_rate=("staleness_days", lambda s: s.notna().mean()),
    mean_impressions=("march_impressions", "mean"),
    median_impressions=("march_impressions", "median"),
)
by_action


,n,staleness_observed_rate,mean_impressions,median_impressions
action,,,,
MONITOR,321248,0.086821,772.966210,1.0
REVIEW_REFRESH,10189,0.999902,3174.378251,1478.0


**The flag-linked test does NOT fully hold.** `REVIEW_REFRESH` pages do have much
higher impressions on average (supports the "meaningful search opportunity" half of the
rule). But staleness is only *observed* for 8.7% of `MONITOR` pages versus 99.99% of
`REVIEW_REFRESH` pages — meaning a page can only realistically reach `REVIEW_REFRESH`
if it happens to have a recorded update date in the first place. Pages with genuinely
unknown update history are, in practice, excluded from ever being flagged
`STALE_HIGH_OPPORTUNITY` regardless of how many impressions they get. **Verdict: MIXED**
— the impressions half of the assumption is confirmed; the staleness half is confounded
by missing data, not a clean behavioral signal.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
print("Practical takeaway, in numbers:")
print("- Impressions dominate the current queue: Spearman ~0.96 with baseline_score.")
print("- Staleness barely moves the score among rows where it IS observed (Spearman ~0.09).")
print(f"- Staleness is simply unknown for {(1 - df['staleness_days'].notna().mean()):.1%} of pages,")
print("  and those pages can't currently reach REVIEW_REFRESH no matter their traffic.")


Practical takeaway, in numbers:
- Impressions dominate the current queue: Spearman ~0.96 with baseline_score.
- Staleness barely moves the score among rows where it IS observed (Spearman ~0.09).
- Staleness is simply unknown for 88.5% of pages,
  and those pages can't currently reach REVIEW_REFRESH no matter their traffic.


**What a content team should take from this:** the current queue is, in practice, a
high-impressions queue with a thin staleness overlay — not a balanced staleness-and-opportunity
score. Before trusting `REVIEW_REFRESH` as "the staleness list," the team should ask why
~88% of pages have no recorded update date, since that missingness — not real freshness —
is currently deciding which high-traffic pages ever get a chance to be flagged.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.